In [2]:
# upload the required libraries
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

In [3]:
# import the data

df = pd.read_csv('property_data.csv')
df.head()

,Unnamed: 0,Property_id,property_type,offer,governorate,bedrooms,bathrooms,rent,location,city,...,is_super_agent,broker_id,floor_plans,is_featured,is_premium,description,is_verified,is_verified.1,is_exclusive,is_exclusive.1
0,27387,7343b863-a2cb-4d85-b2dd-935ea61365ff,Apartment,Residential for Rent,"Capital Governorate, Manama",studio,1,245.0,NaN,NaN,...,True,1031.0,NaN,False,False,A different furnished studio opportunity in Sa...,False,False,False,False
1,18554,f1a36ab5-1601-4127-a7a7-5095f167acca,Apartment,Rent,Capital Governorate,1,1,300.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9295,bf629ae9-a3b2-4c2e-8e0e-b0ffce916fa7,Compound,Rent,Northern Governorate,5,5,1200.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28118,26707746-825c-4d65-b63c-dfaa69421365,Apartment,Residential for Rent,Capital Governorate,3,3,600.0,NaN,NaN,...,False,15.0,NaN,True,False,Spacious 3 Bedroom Apartment for Rent in Seef ...,False,False,False,False
4,689,dbc18ce8-1481-480c-9b66-a42653038408,Apartment,for rent,Muharraq Governorate,1,2,450.0,Al Marsa Floating City,Amwaj Islands,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## EDA & cleaning

In [4]:
# Keep the sqm value in entries such as '4,306 sqft / 400 sqm'.
area = df['area_sqm'].astype('string').str.replace(',', '', regex=False).str.strip()
sqm = area.str.extract(r'(?i)(\d+(?:\.\d+)?)\s*sqm\b', expand=False)
df['area_sqm'] = pd.to_numeric(sqm.fillna(area), errors='coerce').astype(float)
df['area_sqm'].head()

0     43.0
1     65.0
2    400.0
3    160.0
4     80.0
Name: area_sqm, dtype: float64

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32090 entries, 0 to 32089
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         32090 non-null  int64  
 1   Property_id        32090 non-null  str    
 2   property_type      32090 non-null  str    
 3   offer              32090 non-null  str    
 4   governorate        32090 non-null  str    
 5   bedrooms           32078 non-null  str    
 6   bathrooms          32086 non-null  str    
 7   rent               32090 non-null  float64
 8   location           2899 non-null   str    
 9   city               8206 non-null   str    
 10  area_sqm           32090 non-null  float64
 11  listing_date       22564 non-null  str    
 12  Property_id.1      32090 non-null  str    
 13  URL                23884 non-null  str    
 14  include_w_e        23884 non-null  str    
 15  title              23884 non-null  str    
 16  area               23884 non-null

In [6]:
# bedrooms column
df[(df['bedrooms'] == '0')] [(df['bathrooms'] == '2') | (df['bathrooms'] == '3')].index
#df.loc[(df['bedrooms'] == '0') & (df['bathrooms'] == '2'), 'bedrooms'] = '1'

#df.loc[(df['bedrooms'] == '0') & (df['bathrooms'] == '3'), 'bedrooms'] = '2'


C:\Users\TALAL ALOTHMAN\AppData\Local\Temp\ipykernel_17984\1942175892.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[(df['bedrooms'] == '0')] [(df['bathrooms'] == '2') | (df['bathrooms'] == '3')].index


Index([4133, 14595, 28923], dtype='int64')

In [7]:
df[df['bedrooms'] == '0']

,Unnamed: 0,Property_id,property_type,offer,governorate,bedrooms,bathrooms,rent,location,city,...,is_super_agent,broker_id,floor_plans,is_featured,is_premium,description,is_verified,is_verified.1,is_exclusive,is_exclusive.1
523,2218,1cb641c0-a0b1-4af5-a052-f7a70e65b404,Apartment,for rent,Capital Governorate,0,1,160.0,Salmaniya,Manama,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1032,2092,06913900-f07b-42ae-a865-4821d99f75f6,Apartment,for rent,Capital Governorate,0,1,140.0,NaN,Manama,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1065,780,78ff9826-c12d-4fc2-a52c-175e973c450e,Apartment,for rent,Capital Governorate,0,1,200.0,Zinj,Manama,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1105,1420,77880d80-c93f-4e0a-83c1-d1347ccf3c0f,Apartment,for rent,Capital Governorate,0,1,190.0,Zinj,Manama,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1795,7449,e4c49c36-0b8d-409b-9e08-3efdb9156ad2,Apartment,for rent,Muharraq Governorate,0,1,160.0,NaN,Busaiteen,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1886,8300,e35252dc-9199-4dbb-b28d-75f18d404d40,Apartment,for rent,Capital Governorate,0,1,375.0,NaN,Al Juffair,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2396,2084,88c47150-25c0-480f-9263-5e11c63720d4,Apartment,for rent,Capital Governorate,0,1,140.0,Gufool,Manama,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2891,4242,87568cf5-5d93-4fc8-8848-38a192bf6016,Apartment,for rent,Northern Governorate,0,1,230.0,NaN,Saar,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3490,1216,dbca3133-9eab-4624-8086-21f45dba94b2,Apartment,for rent,Capital Governorate,0,1,160.0,Zinj,Manama,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3797,495,764189bb-6180-444e-8914-3bb2851c7948,Apartment,for rent,Muharraq Governorate,0,1,275.0,Amwaj Marina,Amwaj Islands,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df['bedrooms'].unique()

<StringArray>
[      'studio',            '1',            '5',            '3',
            '2',            '4',      '3+ Maid',      '6+ Maid',
      '4+ Maid',      '2+ Maid',      '5+ Maid',            '7',
      '1+ Maid',            '0',           '7+',           '10',
            '6',      '7+ Maid',            nan, 'studio+ Maid',
     '7++ Maid']
Length: 21, dtype: str

In [9]:
X = df[['area_sqm']]
y = df['rent']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [11]:
lr = LinearRegression()

lr.fit(X_train,y_train)

y_hat = lr.predict(X_test)

In [12]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Helper function to evaluate our models (calculates R-squared, RMSE, and MAE for train and test sets)
def evaluate_model(model, X_train, y_train, X_test, y_test):
    results = {
        "Training R-squared": model.score(X_train, y_train),
        "Test R-squared": model.score(X_test, y_test),
        "Training MAE": mean_absolute_error(y_train, model.predict(X_train)),
        "Test MAE": mean_absolute_error(y_test, model.predict(X_test))
    }
    return results

In [13]:
evaluate_model(lr,X_train,y_train,X_test,y_test)

{'Training R-squared': 0.009899663298502892,
 'Test R-squared': 0.0010001785781103,
 'Training MAE': 276.87129965255787,
 'Test MAE': 471.7552222812993}

In [14]:
df_test = pd.read_csv('test.csv')
df_test.head()

,Unnamed: 0,Property_id,property_type,offer,governorate,bedrooms,bathrooms,location,city,area_sqm,...,is_super_agent,broker_id,floor_plans,is_featured,is_premium,description,is_verified,is_verified.1,is_exclusive,is_exclusive.1
0,8192,25c91c33-a523-42f2-8c10-9f340005471d,Apartment,for rent,Capital Governorate,2,3,NaN,Al Juffair,140.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,18541,6bcb7804-9cd7-4792-8cbe-abb4de65058a,Apartment,Rent,Muharraq Governorate,3,3,NaN,NaN,"1,399 sqft / 130 sqm",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5370,7b733e67-ec21-4427-95f8-4be3d9d18f6b,Villa,for rent,Northern Governorate,4,4,NaN,Hamala,400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,33185,feefd51f-a7d6-45b0-a9eb-200d9f9821b4,Apartment,Residential for Rent,"Capital Governorate, Manama",2,2,NaN,NaN,120.0,...,False,1155.0,NaN,False,False,"This Apartment is comprises of 2 bedrooms, 2 b...",False,False,False,False
4,2841,cca286dd-ddc6-41c2-9584-f8e17461220f,Apartment,for rent,Muharraq Governorate,2,3,Amwaj Avenue,Amwaj Islands,130.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Keep the sqm value in entries such as '4,306 sqft / 400 sqm'.
area = df_test['area_sqm'].astype('string').str.replace(',', '', regex=False).str.strip()
sqm = area.str.extract(r'(?i)(\d+(?:\.\d+)?)\s*sqm\b', expand=False)
df_test['area_sqm'] = pd.to_numeric(sqm.fillna(area), errors='coerce').astype(float)
df_test['area_sqm'].head()

0    140.0
1    130.0
2    400.0
3    120.0
4    130.0
Name: area_sqm, dtype: float64

In [16]:
X_sub = df_test[['area_sqm']]

In [17]:
y_hat_sub = lr.predict(X_sub)

In [18]:
submission = pd.DataFrame({
    "Property_id": df_test['Property_id'],
    "price": y_hat_sub
})

submission.to_csv("submission.csv", index=False)